# Free Model Gateway — Colab T4

**Ollama → LiteLLM → Cloudflare Quick Tunnel** (exactly **one** model per runtime).

**Setup:** clone/upload this repo so `config/models.yaml` is visible, then pick `MODEL` / `CONTEXT` / `HOST_ID` and **Runtime → Run all**.

Restart the Colab runtime before switching models (VRAM).

In [ ]:
# @title Configuration
# Models must be listed on the selected host in config/hosts.yaml.
# For 12B/14B on T4, prefer CONTEXT 8192 or 16384.
MODEL = "qwen3-8b"  # @param ["qwen3-4b", "gemma3-4b", "mistral-7b", "llama3-8b", "qwen3-8b", "deepseek-r1-8b", "gemma3-12b", "deepseek-r1-14b", "qwen3-14b"]
CONTEXT = 16384  # @param [4096, 8192, 16384, 32768, 65536]
HOST_ID = "colab-t4-01"  # @param ["colab-t4-01", "colab-t4-02"]

# Optional: set a stable key; otherwise one is generated for this session.
# import os; os.environ["GATEWAY_API_KEY"] = "your-long-secret"

print(f"MODEL={MODEL} CONTEXT={CONTEXT} HOST_ID={HOST_ID}")

In [ ]:
# @title Locate repo + Python path
from pathlib import Path
import sys

# If you uploaded a zip, extract first or set REPO_ROOT manually:
# REPO_ROOT = Path("/content/ai_free_providers")
def _find_repo() -> Path:
    search = [Path.cwd(), *Path.cwd().parents, Path("/content")]
    for candidate in search:
        if (candidate / "config" / "models.yaml").is_file():
            return candidate
        matches = list(candidate.glob("*/config/models.yaml"))
        if matches:
            return matches[0].parent.parent
    raise FileNotFoundError(
        "Upload/clone the repo into Colab so config/models.yaml exists "
        "(e.g. /content/ai_free_providers)."
    )

REPO_ROOT = _find_repo()
sys.path.insert(0, str(REPO_ROOT / "cli"))
sys.path.insert(0, str(REPO_ROOT / "host" / "colab"))
print("REPO_ROOT=", REPO_ROOT)

In [ ]:
# @title Detect GPU
import subprocess

result = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    text=True,
)
print(result)

In [ ]:
# @title Validate selection (registry)
from modelctl.services.registry import load_registry, validate_selection
from modelctl.services.vram import detect_gpus

registry = load_registry(REPO_ROOT / "config")
gpus = detect_gpus()
print("GPU:", gpus[0].name, f"{gpus[0].memory_total_gb:.1f} GB")

if MODEL not in registry.models:
    raise KeyError(f"Unknown MODEL={MODEL!r}. Pick one of: {', '.join(sorted(registry.models))}")

_cfg = registry.models[MODEL]

# Prefer registry default for large models if CONTEXT is still a Phase-1-style 32k
if CONTEXT > _cfg.max_context:
    print(f"WARNING: CONTEXT {CONTEXT} > max_context {_cfg.max_context} → clamp to {_cfg.max_context}")
    CONTEXT = _cfg.max_context
elif _cfg.min_vram_gb >= 11 and CONTEXT > _cfg.default_context:
    print(
        f"NOTE: {MODEL} is large (min_vram={_cfg.min_vram_gb}GB); "
        f"lowering CONTEXT {CONTEXT} → {_cfg.default_context} (safer on T4)"
    )
    CONTEXT = _cfg.default_context

print(
    f"Selection: MODEL={MODEL} ollama={_cfg.model} "
    f"default_ctx={_cfg.default_context} max_ctx={_cfg.max_context} "
    f"min_vram={_cfg.min_vram_gb}GB → using CONTEXT={CONTEXT}"
)

host, model = validate_selection(
    registry,
    HOST_ID,
    MODEL,
    CONTEXT,
    available_vram_gb=gpus[0].memory_total_gb,
)
print("OK:", host.id, model.id, model.model, f"ctx={CONTEXT}")
print("Allowed on host:", ", ".join(host.allowed_models))

# Hint: which registry models fit this GPU (ignore host allow-list)
fit = [
    mid
    for mid, m in sorted(registry.models.items())
    if m.min_vram_gb <= gpus[0].memory_total_gb
]
print("VRAM-fit (any host):", ", ".join(fit))

### If `has install_all: False`

Colab is using a stale `bootstrap.py`. On your PC, upload **only** this file into `/content/ai_free_providers/host/colab/`:

`host/colab/force_sync.py`

Then run:

```python
exec(open("/content/ai_free_providers/host/colab/force_sync.py").read())
```

Expect `has install_all: True`, then re-run the READY cell below.

In [ ]:
# @title Install → start → tunnel → READY
import sys
from pathlib import Path

# Drop cached modules from older uploads
for _name in list(sys.modules):
    if _name in {"bootstrap", "runtime"} or _name.startswith("bootstrap.") or _name.startswith("runtime."):
        del sys.modules[_name]

import bootstrap
import runtime

print("bootstrap file:", getattr(bootstrap, "__file__", "?"))
print("has install_all:", hasattr(bootstrap, "install_all"))

if not hasattr(bootstrap, "install_all"):
    sync = Path("/content/ai_free_providers/host/colab/force_sync.py")
    raise RuntimeError(
        "Colab still has an OLD bootstrap.py.\n"
        "Fix: upload host/colab/force_sync.py from your PC, then run:\n"
        f"  exec(open(r'{sync}').read())\n"
        "Expect: has install_all: True — then re-run this cell."
    )

result = runtime.run(
    model_id=MODEL,
    context=CONTEXT,
    host_id=HOST_ID,
    repo_root=REPO_ROOT,
    skip_install=False,
    run_chat_probe=True,
)
print("endpoint:", result.endpoint)

## Verify from your PC

```bash
export URL="https://xxxxx.trycloudflare.com"   # from READY banner
export GATEWAY_API_KEY="..."                   # printed below banner

curl -sS -H "Authorization: Bearer $GATEWAY_API_KEY" "$URL/v1/models"
```